---
# Example Notebook 02 - WFSS Spectral Extraction

This notebook follows from example notebook 01 and extracts 2D spectra for sources in a source catalogue.

To extract spectra you need the same ancillarry files as for the reduction:
- A segmentation image, ideally from a LW stack
- A detection catalogue from the same LW stack
- Mosaics for every filter you are perfoming extraction on

as well as a source catalogue (this can be different to the detection catalogue).

The extraction and stacking steps are run in parallel, so make sure `N_PROC` is suitable for your hardware.

---



In [6]:
import os
import sys
import glob
import warnings

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table
import multiprocessing as mp


import grizli
from grizli import utils, multifit, fitting, model, grismconf
from grizli.pipeline import auto_script

utils.set_warnings()
warnings.filterwarnings("ignore", category=RuntimeWarning)

WORK_ROOT = "/data/users/cooper/WFSS_GRIZLI_NEW"   
FIELD = "a2744"
EXTRACT_DIR = os.path.join(WORK_ROOT, FIELD, "Extractions")


# Move to extraction directory
os.chdir(EXTRACT_DIR)
utils.LOGFILE = os.path.join(WORK_ROOT, FIELD, f"{FIELD}.extract.log.txt")

# Collect files
flt_files = sorted(glob.glob("*GrismFLT.fits"))
cat_file = f"{FIELD}-ir.cat.fits"
seg_file = f"{FIELD}-ir_seg.fits"

N_PROC = 36 # Change this to work with your system

## Implement some of the same fixes as notebook 01

In [3]:
grismconf.NIRCAM_CONF_VERSION = "V9"

if not hasattr(model.GrismFLT, "_compute_model_orders_orig"):
    model.GrismFLT._compute_model_orders_orig = model.GrismFLT.compute_model_orders

def _safe_compute_model_orders(self, *args, **kwargs):
    try:
        return self._compute_model_orders_orig(*args, **kwargs)
    except KeyError:
        return False

model.GrismFLT.compute_model_orders = _safe_compute_model_orders

if not hasattr(utils, "_log_exception_orig"):
    utils._log_exception_orig = utils.log_exception

def _filtered_log_exception(LOGFILE, traceback, *args, **kwargs):
    if sys.exc_info()[0] is IndexError and "process_config" in traceback.format_exc():
        return
    return utils._log_exception_orig(LOGFILE, traceback, *args, **kwargs)

utils.log_exception = _filtered_log_exception

## 01. Load the target catalog and match to the segmentation image

In [4]:
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u

# Set file information here
TARGET_FILE = "/data/users/cooper/WFSS_GRIZLI_NEW/Catalogues/A2744_coords.csv"     
ID_COL, RA_COL, DEC_COL = "id", "ra", "dec"  

tin = Table.read(TARGET_FILE)
targets = Table()
targets["target_id"] = tin[ID_COL]
targets["ra"] = np.asarray(tin[RA_COL], dtype=float)
targets["dec"] = np.asarray(tin[DEC_COL], dtype=float)

# Identify which sources match a source in the segment image
with fits.open(seg_file, memmap=True) as h:
    seg = h[0].data
    wcs = WCS(h[0].header)
    ny, nx = seg.shape
    x, y = wcs.world_to_pixel_values(targets["ra"], targets["dec"])
    xi, yi = np.round(x).astype(int), np.round(y).astype(int)
    on_mosaic = (xi >= 0) & (xi < nx) & (yi >= 0) & (yi < ny)
    seg_id = np.zeros(len(targets), dtype=int)
    seg_id[on_mosaic] = [seg[j, i] for i, j in zip(xi[on_mosaic], yi[on_mosaic])]

# ---- Nearest catalogue source as a cross-check ------------------------------
cat = Table.read(cat_file)
c_t = SkyCoord(targets["ra"], targets["dec"], unit="deg")
c_c = SkyCoord(cat["RA"], cat["DEC"], unit="deg")
idx, d2d, _ = c_t.match_to_catalog_sky(c_c)

targets["id"] = seg_id                                    
targets["nearest_id"] = np.asarray(cat["NUMBER"])[idx]
targets["nearest_dist"] = d2d.to(u.arcsec).value
targets["nearest_dist"].format = ".2f"
targets["mag"] = np.asarray(cat["MAG_AUTO"])[idx]
targets["mag"].format = ".2f"

keep = targets["id"] >= 0
extract_list = Table()
extract_list["target_id"] = targets["target_id"][keep]
extract_list["grizli_id"] = targets["id"][keep].astype(int)
extract_list["group_name"] = [f"{FIELD}-{t}" for t in extract_list["target_id"]]

## 02. Do the extraction

**Note:** All FLT files are loaded into memory. If you are working on a low RAM machine you may want to extract from a set of files at a time.

In [14]:
flt_files = sorted(glob.glob("*GrismFLT.fits"))
grp = multifit.GroupFLT(grism_files=flt_files, direct_files=[], ref_file=None,
                        seg_file=seg_file, catalog=cat_file,
                        cpu_count=CPU_COUNT, sci_extn=1, pad=(64, 256),
                        use_jwst_crds=False, verbose=False)

print(f"Loaded {len(grp.FLTs)} exposures:")

Load jw03538001001_02101_00001_nrcalong.01.GrismFLT.fits!
Load jw03538001001_02101_00001_nrcblong.01.GrismFLT.fits!
Load jw03538001001_02101_00002_nrcalong.01.GrismFLT.fits!
Load jw03538001001_02101_00002_nrcblong.01.GrismFLT.fits!
Load jw03538001004_02101_00001_nrcalong.01.GrismFLT.fits!





Load jw03538001004_02101_00001_nrcblong.01.GrismFLT.fits!

Load jw03538001004_02101_00002_nrcalong.01.GrismFLT.fits!


Load jw03538001004_02101_00002_nrcblong.01.GrismFLT.fits!




Load jw03538004001_03101_00001_nrcalong.01.GrismFLT.fits!
Load jw03538004001_03101_00001_nrcblong.01.GrismFLT.fits!


Load jw03538004001_03101_00002_nrcalong.01.GrismFLT.fits!


Load jw03538004001_03101_00002_nrcblong.01.GrismFLT.fits!

Load jw03538005001_02101_00001_nrcalong.01.GrismFLT.fits!



Load jw03538005001_02101_00001_nrcblong.01.GrismFLT.fits!


Load jw03538005001_02101_00002_nrcalong.01.GrismFLT.fits!


Load jw03538005001_02101_00002_nrcblong.01.GrismFLT.fits!


Load jw03538005004_02101_00001_nrcalong.01.Gri

In [15]:
SIZE = 72
MIN_SENS = 0.02
MIN_MASK = 0.01
FCONTAM = 0.2
SYS_ERR = 0.03

def extract_one(args):
    tid, gid, gname = args
    out_file = f"{gname}_{gid:05d}.beams.fits"
    if os.path.exists(out_file):
        return (tid, gid, -1, "", "exists")
    try:
        beams = grp.get_beams(gid, size=SIZE, beam_id="A", min_sens=MIN_SENS, min_mask=MIN_MASK)
        if len(beams) == 0:
            return (tid, gid, 0, "", "no coverage")
        mb = multifit.MultiBeam(beams, fcontam=FCONTAM, group_name=gname, psf=False,
                                isJWST=True, sys_err=SYS_ERR,
                                min_mask=MIN_MASK, min_sens=MIN_SENS)
        mb.write_master_fits()
        filts = sorted({b.grism.filter for b in mb.beams})
        return (tid, gid, len(mb.beams), ",".join(filts), "ok")
    except Exception as e:
        return (tid, gid, 0, "", f"error: {e}")

jobs = [(r["target_id"], int(r["grizli_id"]), r["group_name"]) for r in extract_list]
rows = []
with mp.get_context("fork").Pool(N_PROC) as pool:
    for i, res in enumerate(pool.imap_unordered(extract_one, jobs, chunksize=4), start=1):
        rows.append(res)

extract_status = Table(rows=rows, names=["target_id", "grizli_id", "n_beams", "filters", "status"])
print(f"\nDone in {time.time() - t0:.0f} s")
for s in ["ok", "exists", "no coverage"]:
    print(f"  {s:<12}: {np.sum(extract_status['status'] == s)}")
errors = extract_status[[s.startswith("error") for s in extract_status["status"]]]
print(f"  {'errors':<12}: {len(errors)}")
if len(errors):
    errors.pprint(max_lines=20, max_width=-1)

a2744-0_12535.beams.fits
a2744-1_09715.beams.fits
a2744-2_12181.beams.fits
a2744-3_08906.beams.fits
a2744-4_09766.beams.fits
a2744-5_11863.beams.fits
a2744-6_11249.beams.fits
a2744-7_11865.beams.fits
a2744-8_09160.beams.fits
a2744-9_10444.beams.fits
a2744-10_11868.beams.fits
a2744-11_10805.beams.fits
a2744-12_09806.beams.fits
a2744-13_10887.beams.fits
a2744-14_10044.beams.fits
a2744-15_12265.beams.fits
a2744-16_11869.beams.fits
a2744-17_10045.beams.fits
a2744-18_12266.beams.fits
a2744-19_11717.beams.fits
a2744-20_11521.beams.fits
a2744-21_10414.beams.fits
a2744-22_09308.beams.fits
a2744-23_12398.beams.fits
a2744-24_11613.beams.fits
  25/566 targets, 29 s
a2744-25_10376.beams.fits
a2744-26_08921.beams.fits
a2744-27_10866.beams.fits
a2744-28_11525.beams.fits
a2744-29_11690.beams.fits
a2744-30_09042.beams.fits
a2744-31_11275.beams.fits
a2744-32_09441.beams.fits
a2744-33_12729.beams.fits
a2744-34_11084.beams.fits
a2744-35_10502.beams.fits
a2744-36_09595.beams.fits
a2744-37_12686.beams.fits

## 02. Stacking 2D Spectra

I am using grizli's built in tools to stack spectra and produce output figures. All filters are combined into one fits file with a structure as follows:
```
[0] PRIMARY                                      Header only: Overview of the file
For each filter that covers the target:
    For each PA (including different GRISMS)
        SCI    EXTVER = "F410M,<PA>"             2D Spectrum
        WHT    EXTVER = "F410M,<PA>"             2D Weights
        CONTAM EXTVER = "F410M,<PA>"             2D Contamination Model
        MODEL  EXTVER = "F410M,<PA>"             2D Model Spectrum (useful for extracting 1D spectra)
        KERNEL EXTVER = "F410M,<PA>"             Emission line model     
    Combined stack (All GRISMS and PAs):
        SCI    EXTVER = "F410M"
        WHT    EXTVER = "F410M"
        CONTAM EXTVER = "F410M"
        MODEL  EXTVER = "F410M"
        KERNEL EXTVER = "F410M"
```

In [19]:
STACK_SIZE = 32
FLAMBDA = True
KERNEL = "point"
PIXFRAC = 0.5

def stack_one(args):
    tid, gid = args
    gname = f"{FIELD}-{tid}"
    beams_file = f"{gname}_{gid:05d}.beams.fits"
    stack_file = f"{gname}_{gid:05d}.stack.fits"
    if os.path.exists(stack_file):
        return (tid, gid, "", "exists")
    try:
        plt.switch_backend("Agg")            
        mb = multifit.MultiBeam(beams_file, group_name=gname, fcontam=FCONTAM,
                                sys_err=SYS_ERR, min_sens=MIN_SENS, min_mask=MIN_MASK,
                                MW_EBV=MW_EBV)
        hdu, fig = mb.drizzle_grisms_and_PAs(size=STACK_SIZE, fcontam=FCONTAM,
                                             flambda=FLAMBDA, kernel=KERNEL,
                                             pixfrac=PIXFRAC, scale=1, diff=True)
        hdu.writeto(stack_file, overwrite=True)
        fig.savefig(stack_file.replace(".fits", ".png"))
        plt.close(fig)
        return (tid, gid, ",".join(sorted(mb.PA.keys())), "ok")
    except Exception as e:
        return (tid, gid, "", f"error: {e}")

done = extract_status[np.isin(extract_status["status"], ["ok", "exists"])]
jobs = [(r["target_id"], int(r["grizli_id"])) for r in done]
rows = []
t0 = time.time()
with mp.get_context("fork").Pool(N_PROC) as pool:
    for i, res in enumerate(pool.imap_unordered(stack_one, jobs, chunksize=4), start=1):
        rows.append(res)
        if i % 25 == 0:
            print(f"  {i}/{len(jobs)} targets, {time.time() - t0:.0f} s")

stack_status = Table(rows=rows, names=["target_id", "grizli_id", "filters", "status"])
print(f"\nDone in {time.time() - t0:.0f} s")
for s in ["ok", "exists"]:
    print(f"  {s:<7}: {np.sum(stack_status['status'] == s)}")
errors = stack_status[[s.startswith("error") for s in stack_status["status"]]]
print(f"  errors : {len(errors)}")
if len(errors):
    errors.pprint(max_lines=20, max_width=-1)

load_master_fits: a2744-128_14484.beams.fits
load_master_fits: a2744-52_12543.beams.fitsload_master_fits: a2744-84_13319.beams.fitsload_master_fits: a2744-96_13441.beams.fits
load_master_fits: a2744-64_12596.beams.fits
load_master_fits: a2744-148_15317.beams.fits


load_master_fits: a2744-188_16514.beams.fits
load_master_fits: a2744-88_14217.beams.fits
load_master_fits: a2744-56_09942.beams.fits
load_master_fits: a2744-68_12202.beams.fits
load_master_fits: a2744-80_12800.beams.fitsload_master_fits: a2744-124_14485.beams.fits

load_master_fits: a2744-60_13253.beams.fitsload_master_fits: a2744-116_13516.beams.fits

load_master_fits: a2744-112_11889.beams.fitsload_master_fits: a2744-184_09917.beams.fits

load_master_fits: a2744-156_14281.beams.fits
load_master_fits: a2744-120_12572.beams.fitsload_master_fits: a2744-49_13252.beams.fits

load_master_fits: a2744-144_15052.beams.fits
load_master_fits: a2744-100_13481.beams.fitsload_master_fits: a2744-76_13651.beams.fits
  25/552 targets, 2 s
